In [ ]:
import numpy as np
import scipy.constants as constants

from helper_functions import (radial, fourier_shell_correlation, interpolated_intercepts, write_text)

import seaborn as sns
sns.set_theme()

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import h5py
 
import glob, os

e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

showBinDump = False

saveFig = False
multiFSC = True

base_dir = 'average_reconstructions/'
mrc_dir = 'average_reconstructions/chimerax/'

bin_dump_loc = mrc_dir+'binary_dumps/'
    
npy_chimerax = sorted(glob.glob(mrc_dir + '*.npy',recursive = True))
npy_chimerax

<h2> Fourier Shell Correlation </h2> 
Assuming the two 3D Fourier volumes (FT of two 3D electron densities) contain an additive noise component, the FSC for this assumption leads to correct value for zero noise 
case, namely FSC=1, and when there is zero signal that the FSC is equal to the inverse of the number of voxels within each shell used to integrate. 
$$FSC(r_i)\: = \frac{SNR(r_i)+2/\sqrt{n(r_i)}\times\sqrt{SNR(r_i)}+1/\sqrt{n)r_i)}} {SNR(r_i)+2/\sqrt{n(r_i)}\times\sqrt{SNR(r_i)}+1}$$
From this, a threshold curve can now be defined:
$$T_{1.0}(r_i)\: = \frac{0.5+2.4142\times1/\sqrt{n(r_i)}}{1.5+1.4242\times1/\sqrt{n(r_i)}}$$
The previous is actually the 1-bit threshold, at which the reconstruction has reached an SNR of 1.0 for the entire reconstruction (0.5 per half data set). This threshold might be too strict, so the more common
curve is the 0.5-bit threshold, where each reconstruction reaches a SNR of 0.4242 for the entire reconstruction:
$$T_{0.5}(r_i)\: = \frac{0.2071+1.9102\times1/\sqrt{n(r_i)}}{1.2071+0.9102\times1/\sqrt{n(r_i)}}$$
Depending on the symmetry, it might be necessary to divide the $n(r_i)$ by the number of asymmetric units. 
Finally, another measure commonly-used is the $CC_{1/2}$ intensity Pearson correlation coefficient. This measure is more used in X-ray crystallography. Having multiple measures of quality allows us to compare between values reported in different fields. However, having the PRTF and FSC already allows us to provide a graph of resolution versus our parameters and with the resolution defined from two numbers.
Before we plot the FSC, first we set some physical constants in the below cell. Importantly, calculate the maximum scattering angle (corner resolution). 

In [ ]:
# Opening reference volume
name_ref = f'{mrc_dir}'+'1ss8_denss.npy'
vol_ref = np.load(name_ref) # DENSS model

save_name_ref_bin = name_ref.split(sep='/')[1].split(sep='.')[0]
vol_ref_bin = vol_ref.copy()

dim = vol_ref.shape[0]
center = dim//2

if os.path.exists(bin_dump_loc+save_name_ref_bin+'.bin'):
    write_text('Reference volume does exist!\n')
else:
    write_text('Reference volume does not exist!\n')
    vol_ref_bin = vol_ref_bin.astype('f8').tofile(bin_dump_loc+save_name_ref_bin+'.bin')
    
phot_eV = 9000
edge_pixel = 45
d_detector = 1.0 
s_pixel = 2400e-6

phot_m = (h * c) / (phot_eV * e)

theta_edge = 0.5 * np.arctan((edge_pixel*s_pixel)/d_detector) 
edge_res = phot_m/(2.0*np.sin(theta_edge))*1e9
edge_res_inv = 1 / edge_res

write_text(f'Edge resolution: {edge_res} nm\n')
write_text(f'Edge resolution: {edge_res_inv} nm^-1\n')

center_to_corner = np.sqrt((edge_pixel * s_pixel)**2+(edge_pixel * s_pixel)**2)
theta_max = 0.5 * np.arctan(center_to_corner/d_detector)
corner_res = phot_m / (2.0 * np.sin(theta_max)) * 1e9
corner_res_inv = 1 / corner_res

write_text(f'Corner resolution: {corner_res} nm\n')
write_text(f'Corner resolution: {corner_res_inv} nm^-1\n')

s_map_voxel = 1e-10
pix_fourier = 1 / (dim * s_map_voxel * 1e9) # in nm^-1
D_object = 14.7e-9 / s_map_voxe

In [ ]:
for dens_c, dens_name in enumerate(npy_chimerax):
    dens_f = dens_name.split(sep='/')[2]
        
    if dens_f == '1ss8_denss.npy':
        write_text('Reference volume encountered, no FSC calculation!\n')
    else:
        write_text(f'Calculating FSC with reference for: {dens_f}...\n')

        name_alg = f'{mrc_dir}{dens_f}'
        vol_alg = np.load(name_alg) # super-reconstruction density - phased EMC model

        save_name_alg_bin = name_alg.split(sep='/')[2].split(sep='.npy')[0]
        vol_alg_bin = vol_alg.copy()

        if os.path.exists(bin_dump_loc+save_name_alg_bin+'.bin'):
            write_text('Aligned volume does exist!\n')
        else:
            write_text('Aligned volume does not exist!\n')
            vol_alg_bin = vol_alg_bin.astype('f8').tofile(bin_dump_loc+save_name_alg_bin+'.bin')

        fsc, n_ri = fourier_shell_correlation(vol_alg,vol_ref)
        max_points = fsc.shape[0]

        # 0.5-bit curve
        n_ri = n_ri.sum(axis=(1,2,3))
        n_ri_inv = 1 / np.sqrt(n_ri)

        # corrected 0.5-bit curve
        corr_half = True
        c_half = 'no_halfcorr'
        if corr_half:
            n_ri = (n_ri / 2) * (1.5 * (D_object / dim))**2
            n_ri_inv = 1 / np.sqrt(n_ri)
            c_half = 'halfcorr'
            
        half_bit = (0.2071 + 1.9102 * n_ri_inv) / (1.2071 + 0.9102 * n_ri_inv)

        fsc_average = (n_ri*fsc).sum() / n_ri.sum()
        write_text(f'Average FSC: {fsc_average}\n')

        fp_resolution_fsc_inv = np.arange(0, max_points) * pix_fourier

        xci_half, yci_half = interpolated_intercepts(fp_resolution_fsc_inv, fsc, half_bit)
        
        plt.figure(dpi=140)
        plt.plot(fp_resolution_fsc_inv, fsc, c=mcolors.XKCD_COLORS['xkcd:jungle green'], linestyle='-')
        plt.plot(fp_resolution_fsc_inv, half_bit, c='k', linestyle='--')

        if xci_half.size != 0:
            if corr_half:
                if xci_half.size == 1:
                    if 1/xci_half > 10: # don't consider first intersection FSC with threshold
                        write_text(f'Intersection(s) FSC with half-bit criterion: - nm\n')
                    else:
                        plt.plot(xci_half, yci_half, 'ko', ms=7)
                        write_text(f'Intersection(s) FSC with half-bit criterion: {1/xci_half} nm\n')
                else:
                    if 1/xci_half[0] > 10: # don't consider first intersection FSC with threshold
                        plt.plot(xci_half[1:], yci_half[1:], 'ko', ms=7)
                        write_text(f'Intersection(s) FSC with half-bit criterion: {1/xci_half[1:]} nm\n')
            else:
                plt.plot(xci_half, yci_half, 'ko', ms=7)
                write_text(f'Intersection(s) FSC with half-bit criterion: {1/xci_half} nm\n')

        plt.ylim([-0.15, 1.01])
        plt.xlabel('|q| $(nm^{-1})$', weight='bold')
        plt.ylabel('FSC', weight='bold')
        short_name = dens_name.split(sep='/')[-1]
        plt.title(f'{short_name}')
        plt.legend([f'FSC','0.5-bit curve'],frameon=False, prop=dict(weight='bold', size=8), fontsize=0.5, loc=1)

        plt.axvline(x=edge_res_inv, ymin=0 , ymax=1, c=mcolors.XKCD_COLORS['xkcd:blue'],linestyle='--', alpha=0.4) # edge resolution
        plt.axvline(x=corner_res_inv, ymin=0 , ymax=1, c=mcolors.XKCD_COLORS['xkcd:red'],linestyle='--', alpha=0.4) # corner resolution
        plt.axvline(x=fp_resolution_fsc_inv[-1], ymin=0, ymax=1, c=mcolors.XKCD_COLORS['xkcd:green'],linestyle='--', alpha=0.4); # PDB-generated density resolution

        if saveFig:
            with h5py.File(f'figures_fsc/'+save_name_alg_bin+f'_{c_half}_FSC.h5', mode='a') as fsc_handle:
                fsc_handle['fsc_r'] = fsc
                fsc_handle['fp_res_inv_nm'] = fp_resolution_fsc_inv
                if corr_half:
                    if xci_half.size == 1: # one intersection
                        if 1/xci_half < 10:
                            fsc_handle['res_r_nm'] = 1/xci_half
                    elif xci_half.size > 0: # more than one intersection
                        if 1/xci_half[0] > 10:
                            fsc_handle['res_r_nm'] = 1/xci_half[1:]
                    elif xci_half.size == 0: # no intersection
                        pass
                else:
                    fsc_handle['res_r_nm'] = 1/xci_half
                            
                fsc_handle['fsc_average'] = fsc_average
                fsc_handle['n_ri'] = n_ri
                fsc_handle['pdb_reference'] = name_ref
            plt.savefig(f'figures_fsc/'+save_name_alg_bin+f'_{c_half}_FSC.pdf', dpi=150, bbox_inches='tight', pad_inches=0.0);
        write_text('\n')

In [ ]:
files_fsc =  f'figures_fsc/*_prot_wat_*.h5'
if multiFSC:
    filesFSC = np.sort(np.array(glob.glob(files_fsc)))
    names_FSC = []
    combs_FSC = []
    res_FSC = []
    n_ri_inv = []

    i = 0
    for file in filesFSC:
        f_name = file.split(sep='/')[1].split(sep='.h5')[0]
        with h5py.File(file) as f:
            n_ri_inv.append(1 / f['n_ri'][:])
            combs_FSC.append(f['fsc_r'][:])
            res_FSC.append(f['fp_res_inv_nm'][:])
            if 'res_r_nm' in f:
                res_r_nm = f['res_r_nm'][0]
                write_text(f'Resolution for [{i}] {f_name[:-5]}: {res_r_nm} nm\n')
            else:
                write_text(f'Resolution for [{i}] {f_name[:-5]}: - nm\n')
        i += 1
        names_FSC.append(f_name)

    n_ri_inv = np.array(n_ri_inv)
    combs_FSC = np.array(combs_FSC)
    res_FSC = np.array(res_FSC)
    fp_resolution_inv = res_FSC[0,:]
    names_FSC = np.array(names_FSC)
    
    half_bit = (0.2071 + 1.9102 * n_ri_inv) / (1.2071 + 0.9102 * n_ri_inv)
    
    if res_FSC.shape[0] != 0:
        plt.figure(dpi=140)

        plt.axvline(x=edge_res_inv, ymin=0 , ymax=1,c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_') # edge resolution
        plt.axvline(x=corner_res_inv, ymin=0 , ymax=1, c='k', linestyle='--', linewidth=1.0, alpha=0.4, label='_nolegend_') # corner resolution
        plt.plot(fp_resolution_inv, half_bit[-1], c='k', linestyle='--', linewidth=1.0, label='_nolegend_') # DENSS edge resolution

        for p in range(len(filesFSC)):
            plt.plot(fp_resolution_inv, combs_FSC[p], linestyle='-')

        plt.xlim([0.0, fp_resolution_inv[-1]])
        plt.ylim([-0.15, 1.01])
        plt.xlabel('|q| $(nm^{-1})$', weight='bold')
        plt.ylabel('FSC', weight='bold')
        #plt.legend(names_FSC, frameon=True, prop=dict(size=9), loc=1);

<h2> Reading in the binary dump </h2> 
Here we read in both the aligned 3D volume and the reference volume in binary format. We then plot the reference and aligned volumes to distinguish any changes made after the orientation. 
To really ensure the alignment has been done correctly, also plot the absolute difference between aligned and unaligned volumes. 
This will show us visually where the reconstruction differs. Looking at different substructures we can get a feeling of how much 
the reconstruction of various protein regions is affected when changing the number of patterns used for the EMC reconstruction and
the background scaling ratio. 

In [ ]:
if showBinDump:
    vol_ref_plot = np.fromfile('average_reconstructions/chimerax/binary_dumps/1ss8_denss.bin', 
                               dtype='f8').reshape(dim, dim, dim)
    vol_rot_plot = np.fromfile('average_reconstructions/chimerax/binary_dumps/normal_noise_pnccd_1_0001_emc_bg_5_groel_assem_emc_100k_pats_v_1_nsp.bin', 
                               dtype='f8').reshape(dim, dim, dim)
    
    print(f'Shape of reference volume: {vol_ref_plot.shape}')
    print(f'Shape of aligned volume: {vol_rot_plot.shape}')

    fig_handle = plt.figure(constrained_layout = True, dpi = 220)
    fig_handle.patch.set_facecolor(f'white')
    spec_handle = fig_handle.add_gridspec(nrows = 3, ncols = 3)

    zoom = 22
    im_slice = center

    min_v = 0.0
    max_v = None
    cm = 'viridis'

    xy_rot = vol_rot_plot[center-zoom:center+zoom,center-zoom:center+zoom,im_slice]
    xz_rot = vol_rot_plot[center-zoom:center+zoom,im_slice,center-zoom:center+zoom]
    yz_rot = vol_rot_plot[im_slice,center-zoom:center+zoom,center-zoom:center+zoom]
    xc, yc = xy_rot.shape[0]//2, xy_rot.shape[1]//2

    ax_0 = fig_handle.add_subplot(spec_handle[0,0]) 
    im_0 = plt.imshow(xy_rot,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_0.set_title(f'Rot vol XY-{im_slice}/{dim}',weight='bold',fontsize=6)
    ax_0.set_xticks([]) 
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim() 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.3,shrink=0.5,orientation='horizontal') 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1])
    im_1 = plt.imshow(xz_rot,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_1.set_title(f'Rot vol XZ-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_rot,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_2.set_title(f'Rot vol YZ-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_2.set_xticks([])
    ax_2.set_yticks([]) 

    xy_ori = vol_ref_plot[center-zoom:center+zoom,center-zoom:center+zoom,im_slice]
    xz_ori = vol_ref_plot[center-zoom:center+zoom,im_slice,center-zoom:center+zoom]
    yz_ori = vol_ref_plot[im_slice,center-zoom:center+zoom,center-zoom:center+zoom]

    ax_3 = fig_handle.add_subplot(spec_handle[1,0]) 
    im_3 = plt.imshow(xy_ori,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_3.set_title(f'Ref vol XY-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_3.set_xticks([]) 
    ax_3.set_yticks([]) 
    minv, maxv = im_3.get_clim() 
    c_bar_3 = plt.colorbar(im_3, ax=ax_3,fraction=0.3,shrink=0.5,orientation='horizontal') 
    c_bar_3.set_ticks([minv,maxv]) 

    ax_4 = fig_handle.add_subplot(spec_handle[1,1]) 
    im_4 = plt.imshow(xz_ori,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_4.set_title(f'Ref vol XZ-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_4.set_xticks([]) 
    ax_4.set_yticks([]) 

    ax_5 = fig_handle.add_subplot(spec_handle[1,2]) 
    im_5 = plt.imshow(yz_ori,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_5.set_title(f'Ref vol YZ-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_5.set_xticks([]) 
    ax_5.set_yticks([]) 

    xy_diff = np.abs(xy_ori-xy_rot)
    xz_diff = np.abs(xz_ori-xz_rot)
    yz_diff = np.abs(50*yz_ori-yz_rot)

    ax_6 = fig_handle.add_subplot(spec_handle[2,0])
    im_6 = plt.imshow(xy_diff,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_6.set_title(f'Diff vol XY-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_6.set_xticks([]) 
    ax_6.set_yticks([]) 
    minv, maxv = im_6.get_clim() 
    c_bar_6 = plt.colorbar(im_6, ax=ax_6,fraction=0.3,shrink=0.5,orientation='horizontal') 
    c_bar_6.set_ticks([minv,maxv]) 

    ax_7 = fig_handle.add_subplot(spec_handle[2,1])
    im_7 = plt.imshow(xz_diff,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_7.set_title(f'Diff vol XZ-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_7.set_xticks([]) 
    ax_7.set_yticks([]) 

    ax_8 = fig_handle.add_subplot(spec_handle[2,2]) 
    im_8 = plt.imshow(yz_diff,vmin=min_v,vmax=max_v,cmap=cm,interpolation=None)
    plt.plot(xc, yc, 'ro')
    ax_8.set_title(f'Diff vol YZ-{im_slice}/{dim}',weight='bold',fontsize=6) 
    ax_8.set_xticks([])
    ax_8.set_yticks([]);